In [ ]:
import os
import sys

import pandas as pd

#import the same config the solver uses, so days/weekends can never drift
sys.path.insert(0, os.path.abspath(".."))
from src.infrastructure.config import DEFAULT_CONFIG as config

In [ ]:
employees = config.employees
days = config.days
weekends = config.weekends   # derived from the calendar, not hand-maintained
shifts = config.shifts
weekend_indices = config.weekend_indices

print(f"{config.num_employees} employees, {config.num_days} days, {len(weekend_indices)} weekend days")

In [ ]:
path = os.path.join("..", "output", "schedule.csv")

df = pd.read_csv(path)
df.shape

In [ ]:
#column 0 is the employee id, so day d lives in column d + 1
weekend_df = df.iloc[:, [i + 1 for i in weekend_indices]]
weekend_df

In [ ]:
#weekend days worked per employee, against the configured bounds
worked = (weekend_df != "R").sum(axis=1)

print(f"fair share: {config.fair_weekend_days}, hard cap: {config.max_weekend_days_worked}")
print(f"min: {worked.min()}, max: {worked.max()}")
worked.value_counts().sort_index()

In [ ]:
#total shifts per employee, against the fair band
fair_min, fair_max = config.fair_shifts
total = (df.iloc[:, 1:] != "R").sum(axis=1)

print(f"fair band: {fair_min} to {fair_max}")
print(f"outside the band: {((total < fair_min) | (total > fair_max)).sum()} employees")
total.value_counts().sort_index()

In [ ]:
#coverage per day, against min_ee / max_ee per working shift
for s in shifts[:-1]:
    per_day = (df.iloc[:, 1:] == s).sum()
    print(f"{s}: {per_day.min()} to {per_day.max()} (allowed {config.min_ee} to {config.max_ee})")